# 03 Factory Method (Metoda Fabrykujaca) | _Kamil Bartocha_ | wersja 2.0

## Rozklad jazdy

1. ❓ Problem: tworzenie bez znajomosci klasy
2. 🏗️ Uczestnicy: Creator i Product
3. 🐍 Implementacja z `abc.ABC`
4. 📋 Rejestr fabryk
5. 🔧 `@classmethod` jako fabryka

## 1. 🔹 Problem: tworzenie bez znajomosci klasy

Factory Method to wzorzec kreacyjny, ktory definiuje interfejs
do tworzenia obiektow, ale pozwala podklasom decydowac o tym,
jaka klase instancjonowac.

Problem bez wzorca: kod klienta zawiera konkretne klasy (`EmailNotif`,
`SMSNotif`). Dodanie nowego kanalu wymaga modyfikacji klienta -
naruszenie OCP.

Rozwiazanie: klasa Creator definiuje metode fabrykujaca
`create_product()`. Podklasy nadpisuja te metode, zeby zwrocic
wlasciwy konkretny produkt.

> 💡 Factory Method rozdziela kod uzywajacy obiektow od kodu
> tworzacego obiekty. Klient zna tylko interfejs - nie zna
> konkretnych klas.

In [ ]:
# Problem: klient zna konkretne klasy - trudno rozszerzyc
class EmailAlert:
    def send(self, msg: str): print(f'Email: {msg}')

class SMSAlert:
    def send(self, msg: str): print(f'SMS: {msg}')

class AlertSystemBad:
    def alert(self, channel: str, msg: str) -> None:
        if channel == 'email':
            EmailAlert().send(msg)   # konkretna klasa wbudowana
        elif channel == 'sms':
            SMSAlert().send(msg)     # konkretna klasa wbudowana
        # dodanie 'push' wymaga zmiany tej metody - naruszenie OCP!

bad = AlertSystemBad()
bad.alert('email', 'Server down!')  # Email: Server down!
bad.alert('sms', 'Server down!')    # SMS: Server down!

---

### 🐍 Cwiczenia - problem

1. Policz ile `elif` musisz dodac do `AlertSystemBad` zeby obslugiwac
   5 kanalow (email, sms, push, slack, webhook).
2. Napisz `LoggerBad` z metoda `log(target, msg)` i `if/elif` dla
   'file', 'console', 'db'. Policz galezi kodu.
3. *(Trudniejsze)* Zidentyfikuj co sie stanie z `LoggerBad` gdy
   jeden z celow logger przestanie byc uzywany - ile miejsc
   w kodzie musisz zmodyfikowac?

In [ ]:
# Cwiczenie 1: liczba elif dla 5 kanalow
channels = ['email', 'sms', 'push', 'slack', 'webhook']
print(f'Liczba galezi elif: {len(channels) - 1}')
print(f'Kazde nowe kanalowe -> 1 modyfikacja AlertSystemBad')

In [ ]:
# Cwiczenie 2: LoggerBad
class LoggerBad:
    def log(self, target: str, msg: str) -> None:
        if target == 'console':
            print(f'CONSOLE: {msg}')
        elif target == 'file':
            print(f'FILE: {msg}')   # symulacja zapisu
        elif target == 'db':
            print(f'DB: {msg}')     # symulacja zapisu

logger = LoggerBad()
logger.log('console', 'App started')
logger.log('file', 'Error occurred')
print(f'Galezi if/elif: 3 (wzrosnie liniowo z liczbą celow)')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: analiza zmian
analiza = {
    'usun target db': [
        'Usun galaz elif w log()',
        'Upewnij sie ze nikt nie wywoluje log("db", ...)',
        'Zaktualizuj dokumentacje',
    ],
    'dodaj target redis': [
        'Dodaj elif w log()',
        'Napisz logike dla Redis',
    ]
}
for operacja, kroki in analiza.items():
    print(f'{operacja}: {len(kroki)} krokow')
    for k in kroki:
        print(f'  - {k}')

## 2. 🔹 Uczestnicy: Creator i Product

Wzorzec Factory Method ma cztery uczestnikow:

| Uczestnik | Opis | Python |
|---|---|---|
| Product (Produkt) | Interfejs tworzonych obiektow | `abc.ABC` |
| ConcreteProduct | Konkretna implementacja | podklasa |
| Creator (Tworca) | Deklaruje metode fabrykujaca | `abc.ABC` |
| ConcreteCreator | Implementuje metode fabrykujaca | podklasa |

Creator czesto zawiera takze "metode szablonowa" (Template Method),
ktora korzysta z produktu zwroconego przez fabryke.

> 💡 Creator nie musi byc abstrakcyjny - moze miec domyslna
> implementacje metody fabrykujacej, ktora podklasy moga nadpisac.

In [ ]:
from abc import ABC, abstractmethod

# Product - interfejs produktu
class Transport(ABC):
    @abstractmethod
    def deliver(self, cargo: str) -> str: ...

# ConcreteProducts
class Truck(Transport):
    def deliver(self, cargo: str) -> str:
        return f'Truck delivering {cargo} by road'

class Ship(Transport):
    def deliver(self, cargo: str) -> str:
        return f'Ship delivering {cargo} by sea'

class Plane(Transport):
    def deliver(self, cargo: str) -> str:
        return f'Plane delivering {cargo} by air'

# Creator - abstrakcyjny tworca
class Logistics(ABC):
    @abstractmethod
    def create_transport(self) -> Transport: ...  # metoda fabrykujaca

    def plan_delivery(self, cargo: str) -> str:   # metoda szablonowa
        transport = self.create_transport()       # uzywa fabryki
        return transport.deliver(cargo)

# ConcreteCreators
class RoadLogistics(Logistics):
    def create_transport(self) -> Transport:
        return Truck()

class SeaLogistics(Logistics):
    def create_transport(self) -> Transport:
        return Ship()

class AirLogistics(Logistics):
    def create_transport(self) -> Transport:
        return Plane()

# Klient uzywa tylko interfejsu Creator
for logistics in [RoadLogistics(), SeaLogistics(), AirLogistics()]:
    print(logistics.plan_delivery('electronics'))

---

### 🐍 Cwiczenia - Creator i Product

1. Napisz `Button(ABC)` z metoda `click() -> str` i dwie implementacje:
   `WindowsButton`, `MacButton`. Napisz `DialogCreator(ABC)` z metoda
   `create_button()` i dwa konkretne kreatory.
2. Dodaj do `DialogCreator` metode `render()` wywolujaca
   `create_button().click()` - oto metoda szablonowa.
3. *(Trudniejsze)* Napisz `ConcreteCreator` dla systemu Linux.
   Upewnij sie, ze `DialogCreator.render()` dziala bez zmian.

In [ ]:
# Cwiczenie 1: Button i DialogCreator
from abc import ABC, abstractmethod

class Button(ABC):
    @abstractmethod
    def click(self) -> str: ...

class WindowsButton(Button):
    def click(self) -> str: ...

class MacButton(Button):
    def click(self) -> str: ...

class DialogCreator(ABC):
    @abstractmethod
    def create_button(self) -> Button: ...

class WindowsDialogCreator(DialogCreator):
    def create_button(self) -> Button: ...

class MacDialogCreator(DialogCreator):
    def create_button(self) -> Button: ...

print(WindowsDialogCreator().create_button().click())
print(MacDialogCreator().create_button().click())

In [ ]:
# Cwiczenie 2: metoda szablonowa render()
class DialogCreator(ABC):
    @abstractmethod
    def create_button(self) -> Button: ...

    def render(self) -> str:  # metoda szablonowa
        btn = self.create_button()
        return f'Rendering dialog with: {btn.click()}'

class WindowsDialogCreator(DialogCreator):
    def create_button(self) -> Button: return WindowsButton()

class MacDialogCreator(DialogCreator):
    def create_button(self) -> Button: return MacButton()

print(WindowsDialogCreator().render())
print(MacDialogCreator().render())

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: Linux bez zmian w render()
class LinuxButton(Button):
    def click(self) -> str: return 'Linux button clicked'

class LinuxDialogCreator(DialogCreator):
    def create_button(self) -> Button: ...

# render() dziala bez modyfikacji
for creator in [WindowsDialogCreator(), MacDialogCreator(), LinuxDialogCreator()]:
    print(creator.render())

## 3. 🔹 Implementacja z `abc.ABC`

Klasyczna implementacja Factory Method w Pythonie opiera sie
na `abc.ABC` i dekoratorze `@abstractmethod`.

Kroki:
1. Zdefiniuj abstrakcyjny produkt (ABC z `@abstractmethod`)
2. Napisz konkretne produkty dziedziczace po produkcie
3. Zdefiniuj abstrakcyjny kreator z metoda `create_product()`
4. Napisz konkretne kreatory, nadpisujac `create_product()`

Python nie ma slowa kluczowego `interface` - uzywamy ABC lub Protocol.
ABC lepiej tutaj bo zapewniamy wspolny interfejs przez dziedziczenie.

In [ ]:
from abc import ABC, abstractmethod

class Parser(ABC):
    @abstractmethod
    def parse(self, text: str) -> dict: ...

    @abstractmethod
    def supported_format(self) -> str: ...

class JSONParser(Parser):
    def parse(self, text: str) -> dict:
        import json
        return json.loads(text)
    def supported_format(self) -> str:
        return 'application/json'

class CSVParser(Parser):
    def parse(self, text: str) -> dict:
        lines = text.strip().split('\n')
        headers = lines[0].split(',')
        values = lines[1].split(',')
        return dict(zip(headers, values))
    def supported_format(self) -> str:
        return 'text/csv'

class DataImporter(ABC):
    @abstractmethod
    def create_parser(self) -> Parser: ...

    def import_data(self, text: str) -> dict:
        parser = self.create_parser()
        print(f'Using: {parser.supported_format()}')
        return parser.parse(text)

class JSONImporter(DataImporter):
    def create_parser(self) -> Parser:
        return JSONParser()

class CSVImporter(DataImporter):
    def create_parser(self) -> Parser:
        return CSVParser()

print(JSONImporter().import_data('{"name": "Alice", "age": "30"}'))
print(CSVImporter().import_data('name,age\nAlice,30'))

---

### 🐍 Cwiczenia - implementacja z ABC

1. Napisz `Serializer(ABC)` z `serialize(obj: dict) -> str` i dwie
   implementacje: `JSONSerializer`, `XMLSerializer`.
2. Napisz `DataExporter(ABC)` z `create_serializer()` i metoda
   `export(data)` wywolujaca serializer.
3. *(Trudniejsze)* Dodaj do `Serializer` metode `deserialize(text) -> dict`.
   Przetestuj roundtrip: `serialize -> deserialize` dla JSON.

In [ ]:
# Cwiczenie 1: Serializer z ABC
from abc import ABC, abstractmethod
import json

class Serializer(ABC):
    @abstractmethod
    def serialize(self, obj: dict) -> str: ...

class JSONSerializer(Serializer):
    def serialize(self, obj: dict) -> str: ...

class XMLSerializer(Serializer):
    def serialize(self, obj: dict) -> str:
        # format: <root><key>value</key></root>
        ...

data = {'name': 'Alice', 'age': 30}
print(JSONSerializer().serialize(data))
print(XMLSerializer().serialize(data))

In [ ]:
# Cwiczenie 2: DataExporter
class DataExporter(ABC):
    @abstractmethod
    def create_serializer(self) -> Serializer: ...

    def export(self, data: dict) -> str:
        ...

class JSONExporter(DataExporter):
    def create_serializer(self) -> Serializer: ...

class XMLExporter(DataExporter):
    def create_serializer(self) -> Serializer: ...

print(JSONExporter().export({'product': 'Widget', 'price': 9.99}))
print(XMLExporter().export({'product': 'Widget', 'price': 9.99}))

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: roundtrip serialize/deserialize
import json

class Serializer(ABC):
    @abstractmethod
    def serialize(self, obj: dict) -> str: ...
    @abstractmethod
    def deserialize(self, text: str) -> dict: ...

class JSONSerializer(Serializer):
    def serialize(self, obj: dict) -> str: ...
    def deserialize(self, text: str) -> dict: ...

s = JSONSerializer()
original = {'user': 'Bob', 'score': 42}
serialized = s.serialize(original)
restored = s.deserialize(serialized)
print(f'Original:  {original}')
print(f'Roundtrip: {restored}')
print(f'Equal: {original == restored}')

## 4. 🔹 Rejestr fabryk

Rejestr fabryk to bardziej pythonowe podejscie: slownik mapujacy
klucze (np. napisy) na klasy lub callables. Zamiast hierarchii
klas Creator, uzywamy jednej funkcji fabrykujacej.

Zalety rejestru:
- Prosta, czytelna implementacja
- Latwe dodawanie nowych typow (jeden wpis w slowniku)
- Mozna wczytywac z konfiguracji (np. YAML)

Wady rejestru vs. hierarchia:
- Brak sprawdzania w czasie kompilacji przez mypy
- Trudniejsza rozszerzalnosc dla zewnetrznych bibliotek

> 💡 Rejestr fabryk jest czesto lepszym wyborem niz pelna hierarchia
> Creator dla prostych przypadkow w Pythonie.

In [ ]:
from abc import ABC, abstractmethod

class Compressor(ABC):
    @abstractmethod
    def compress(self, data: bytes) -> bytes: ...
    @abstractmethod
    def decompress(self, data: bytes) -> bytes: ...

class ZlibCompressor(Compressor):
    def compress(self, data: bytes) -> bytes:
        import zlib
        return zlib.compress(data)
    def decompress(self, data: bytes) -> bytes:
        import zlib
        return zlib.decompress(data)

class GzipCompressor(Compressor):
    def compress(self, data: bytes) -> bytes:
        import gzip
        return gzip.compress(data)
    def decompress(self, data: bytes) -> bytes:
        import gzip
        return gzip.decompress(data)

# Rejestr fabryk - slownik klas
_COMPRESSOR_REGISTRY: dict[str, type[Compressor]] = {
    'zlib': ZlibCompressor,
    'gzip': GzipCompressor,
}

def get_compressor(algorithm: str) -> Compressor:
    cls = _COMPRESSOR_REGISTRY.get(algorithm)
    if cls is None:
        available = ', '.join(_COMPRESSOR_REGISTRY)
        raise ValueError(f'Unknown: {algorithm}. Available: {available}')
    return cls()

data = b'Hello, World! ' * 100
for algo in ['zlib', 'gzip']:
    comp = get_compressor(algo)
    compressed = comp.compress(data)
    restored = comp.decompress(compressed)
    print(f'{algo}: {len(data)}B -> {len(compressed)}B, OK: {data == restored}')

---

### 🐍 Cwiczenia - rejestr fabryk

1. Napisz rejestr klas formatujacych wyniki: `TableFormatter`,
   `JSONFormatter`, `CSVFormatter` - kazdy ma `format(rows: list) -> str`.
   Funkcja `get_formatter(fmt) -> Formatter` pobiera z rejestru.
2. Dodaj rejestracje przez dekorator `@register_formatter('html')`.
3. *(Trudniejsze)* Zaimplementuj rejestr z domyslnym formatem
   i mozliwoscia wczytania formatu z `os.environ['OUTPUT_FORMAT']`.

In [ ]:
# Cwiczenie 1: rejestr formaterow
from abc import ABC, abstractmethod
import json

class Formatter(ABC):
    @abstractmethod
    def format(self, rows: list[dict]) -> str: ...

class TableFormatter(Formatter):
    def format(self, rows: list[dict]) -> str: ...

class JSONFormatter(Formatter):
    def format(self, rows: list[dict]) -> str: ...

FORMATTER_REGISTRY: dict[str, type[Formatter]] = {
    'table': TableFormatter,
    'json': JSONFormatter,
}

def get_formatter(fmt: str) -> Formatter:
    ...

rows = [{'name': 'Alice', 'score': 95}, {'name': 'Bob', 'score': 87}]
print(get_formatter('json').format(rows))
print(get_formatter('table').format(rows))

In [ ]:
# Cwiczenie 2: dekorator rejestrujacy
def register_formatter(name: str):
    def decorator(cls):
        FORMATTER_REGISTRY[name] = cls
        return cls
    return decorator

@register_formatter('html')
class HTMLFormatter(Formatter):
    def format(self, rows: list[dict]) -> str:
        ...

print(get_formatter('html').format(rows))
print('Dostepne formaty:', list(FORMATTER_REGISTRY.keys()))

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: domyslny format z ENV
import os

DEFAULT_FORMAT = 'json'

def get_default_formatter() -> Formatter:
    # hint: os.environ.get('OUTPUT_FORMAT', DEFAULT_FORMAT)
    fmt = os.environ.get('OUTPUT_FORMAT', DEFAULT_FORMAT)
    return get_formatter(fmt)

os.environ['OUTPUT_FORMAT'] = 'table'
print(f'Format z ENV: {type(get_default_formatter()).__name__}')

del os.environ['OUTPUT_FORMAT']
print(f'Domyslny format: {type(get_default_formatter()).__name__}')

## 5. 🔹 `@classmethod` jako fabryka

`@classmethod` to najprostszy sposob na Factory Method w Pythonie:
alternatywny konstruktor, ktory konwertuje dane z roznych zrodel.

Convencja nazywania: `from_xxx()` gdzie `xxx` opisuje zrodlo:
- `from_string()` - parsowanie napisu
- `from_dict()` - z slownika
- `from_file()` - z pliku
- `from_env()` - ze zmiennych srodowiskowych

Przyklady z biblioteki standardowej:
- `datetime.fromisoformat('2024-01-15')`
- `datetime.fromtimestamp(1705312200)`
- `int.from_bytes(b'\x01\x00', 'big')`
- `dict.fromkeys(['a', 'b', 'c'], 0)`

> 💡 `@classmethod` jako fabryka to idiom Pythona. Uzywaj go gdy
> masz kilka sposobow tworzenia obiektu - zamiast skomplikowanego
> `__init__` z wieloma opcjonalnymi parametrami.

In [ ]:
import json
from datetime import datetime

class Config:
    def __init__(self, host: str, port: int, debug: bool, timeout: int):
        self.host = host
        self.port = port
        self.debug = debug
        self.timeout = timeout

    @classmethod
    def from_dict(cls, d: dict) -> 'Config':
        return cls(
            host=d.get('host', 'localhost'),
            port=int(d.get('port', 8080)),
            debug=bool(d.get('debug', False)),
            timeout=int(d.get('timeout', 30)),
        )

    @classmethod
    def from_json_string(cls, s: str) -> 'Config':
        return cls.from_dict(json.loads(s))

    @classmethod
    def from_env(cls) -> 'Config':
        import os
        return cls(
            host=os.getenv('APP_HOST', 'localhost'),
            port=int(os.getenv('APP_PORT', '8080')),
            debug=os.getenv('APP_DEBUG', '').lower() == 'true',
            timeout=int(os.getenv('APP_TIMEOUT', '30')),
        )

    @classmethod
    def development(cls) -> 'Config':
        return cls('localhost', 5000, True, 5)

    @classmethod
    def production(cls) -> 'Config':
        return cls('0.0.0.0', 80, False, 60)

    def __repr__(self) -> str:
        return f'Config({self.host}:{self.port}, debug={self.debug})'

cfg1 = Config.from_dict({'host': 'db.prod', 'port': '5432', 'debug': 'false', 'timeout': '60'})
cfg2 = Config.from_json_string('{"host": "redis", "port": 6379, "debug": false, "timeout": 10}')
cfg3 = Config.development()
cfg4 = Config.production()

for c in [cfg1, cfg2, cfg3, cfg4]:
    print(c)

# Stdlib: datetime.fromisoformat, datetime.fromtimestamp
now = datetime.fromisoformat('2024-01-15T10:30:00')
print(f'datetime: {now}')

---

### 🐍 Cwiczenia - @classmethod jako fabryka

1. Napisz klase `Color` z polami `r, g, b` (int 0-255). Dodaj fabryki:
   `from_hex('#ff8800')` i `from_name('red')` (slownik kilku kolorow).
2. Napisz klase `Money` z polami `amount` (Decimal) i `currency` (str).
   Dodaj `from_string('100.50 USD')` parsujaca napis.
3. *(Trudniejsze)* Napisz klase `DateRange` z polami `start`, `end`
   (datetime). Dodaj `last_n_days(n: int)`, `this_month()`,
   `from_strings(start_str, end_str)` uzywajac `datetime.fromisoformat`.

In [ ]:
# Cwiczenie 1: Color z fabrykami
class Color:
    NAMED = {'red': (255,0,0), 'green': (0,255,0), 'blue': (0,0,255)}

    def __init__(self, r: int, g: int, b: int):
        self.r, self.g, self.b = r, g, b

    @classmethod
    def from_hex(cls, hex_str: str) -> 'Color':
        # hint: hex_str[1:3], hex_str[3:5], hex_str[5:7] -> int(x, 16)
        ...

    @classmethod
    def from_name(cls, name: str) -> 'Color':
        ...

    def __repr__(self) -> str:
        return f'Color(r={self.r}, g={self.g}, b={self.b})'

print(Color.from_hex('#ff0000'))    # Color(r=255, g=0, b=0)
print(Color.from_name('green'))    # Color(r=0, g=255, b=0)

In [ ]:
# Cwiczenie 2: Money z from_string
from decimal import Decimal

class Money:
    def __init__(self, amount: Decimal, currency: str):
        self.amount = amount
        self.currency = currency

    @classmethod
    def from_string(cls, s: str) -> 'Money':
        # format: '100.50 USD'
        ...

    def __repr__(self) -> str:
        return f'Money({self.amount} {self.currency})'

print(Money.from_string('100.50 USD'))  # Money(100.50 USD)
print(Money.from_string('49.99 EUR'))   # Money(49.99 EUR)

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: DateRange z fabrykami
from datetime import datetime, timedelta

class DateRange:
    def __init__(self, start: datetime, end: datetime):
        self.start = start
        self.end = end

    @classmethod
    def last_n_days(cls, n: int) -> 'DateRange':
        # hint: datetime.now() - timedelta(days=n)
        ...

    @classmethod
    def this_month(cls) -> 'DateRange':
        # hint: replace(day=1) dla pierwszego dnia, koniec miesiaca
        ...

    @classmethod
    def from_strings(cls, start_str: str, end_str: str) -> 'DateRange':
        ...

    def __repr__(self) -> str:
        fmt = '%Y-%m-%d'
        return f'DateRange({self.start.strftime(fmt)} - {self.end.strftime(fmt)})'

print(DateRange.last_n_days(7))
print(DateRange.from_strings('2024-01-01', '2024-01-31'))